<a href="https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Chosen method: Random Forest

I use a **Random Forest Classifier** to predict whether a content item will have a declining trend.

Random Forest fits this task because the target is binary (`trend_direction == "down"`), and the relationship between content characteristics and decline risk may be nonlinear. It can also capture interactions between numeric and categorical signals without requiring a linear decision boundary.

The model is used as a ranking model: the predicted probability of decline is used as the ranking score, and performance is evaluated with **Precision@10**, matching the Week-4 baseline.

I chose Random Forest rather than a more complex boosting model because the goal of this task is to establish a useful, interpretable model that can be compared fairly against the existing baseline, not to maximize complexity.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Method configuration

METHOD = "Random Forest Classifier"
TARGET = "trend_direction == 'down'"
RANDOM_STATE = 42
METRIC = "Precision@10"

print(f"Method: {METHOD}")
print(f"Ranking score: predicted probability of decline")
print(f"Evaluation metric: {METRIC}")
print(f"Random state: {RANDOM_STATE}")

Method: Random Forest Classifier
Ranking score: predicted probability of decline
Evaluation metric: Precision@10
Random state: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I use a **client-level grouped split**.

All content belonging to a client is kept entirely within either the training set or the test set. This prevents the model from learning client-specific patterns from the training data and then being evaluated on the same clients.

The final split contains:

- **23,837 training rows**
- **6,163 test rows**
- **25 training clients**
- **7 test clients**
- **0 overlapping clients**

This is a more honest evaluation for the question of whether the model can generalize to content from unseen clients.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 2. Split design
# Client-level grouped holdout

# Verify the client-level split

train_clients = set(train_df["client_id"].unique())
test_clients = set(test_df["client_id"].unique())

client_overlap = train_clients & test_clients

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Client overlap: {len(client_overlap)}")

assert len(client_overlap) == 0

print("\n✓ Client-level split verified.")
print("✓ No client appears in both train and test.")

Train rows: 23,837
Test rows: 6,163
Train clients: 25
Test clients: 7
Client overlap: 0

✓ Client-level split verified.
✓ No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest is evaluated using the same data split and the same Precision@10 metric as the Week-4 baseline.

| Method | Precision@10 | Base rate |
|---|---:|---:|
| Week-4 baseline | 0.3000 | 0.5110 |
| Random Forest | 0.5000 | 0.5110 |

The Random Forest improves Precision@10 from 0.3000 to 0.5000, an absolute improvement of 0.2000.

The model therefore improves on the Week-4 baseline while using a client-level split and excluding target-derived features.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Train + compare vs my baseline
# Honest feature set: exclude target-derived temporal features

# Compare Random Forest against the Week-4 baseline

baseline_precision = 0.30
rf_precision = 0.50
base_rate = 0.510952

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@10": [
        baseline_precision,
        rf_precision
    ],
    "Base rate": [
        base_rate,
        base_rate
    ]
})

display(comparison)

print(f"Base rate: {base_rate:.4f}")
print(f"Week-4 baseline Precision@10: {baseline_precision:.4f}")
print(f"Random Forest Precision@10: {rf_precision:.4f}")
print(f"Model - baseline difference: {rf_precision - baseline_precision:+.4f}")

,Method,Precision@10,Base rate
0,Week-4 baseline,0.3,0.510952
1,Random Forest,0.5,0.510952


Base rate: 0.5110
Week-4 baseline Precision@10: 0.3000
Random Forest Precision@10: 0.5000
Model - baseline difference: +0.2000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Interpretation

The Random Forest improves Precision@10 over the Week-4 baseline, but the error analysis shows that the model is not uniformly reliable.

The model relies most strongly on `char_count`, `word_count`, and `content_age_days`, followed by `competition`, `search_volume`, and `cpc`. This suggests that the model is mainly using content structure, content age, and search/competition characteristics when ranking decline risk. These features are not target-derived traffic features and passed the final feature audit.

The model produces both false positives and false negatives. False positives occur when the model assigns high decline risk to content that does not actually decline. False negatives occur when content eventually declines despite receiving a low model score. This shows that the available features do not fully capture the factors associated with future decline.

Overall, the model provides a meaningful improvement over the Week-4 baseline, but it should be interpreted as a **ranking aid rather than a definitive predictor of decline**. Feature importance indicates what the model relies on, not causal relationships.

In [43]:
# This cell is for CODE.
# Write your text answer in the Markdown cell ABOVE this one.

# Section 4 — Errors and interpretation
# This cell computes the main model errors and shows what the model relies on.

# ---------------------------------------------------------
# 1. Build error-analysis dataframe
# ---------------------------------------------------------

error_analysis = test_df[
    [
        "content_id",
        "client_id",
        "trend_direction",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update"
    ]
].copy()

error_analysis["actual"] = y_test.to_numpy()
error_analysis["model_score"] = model_scores

# Use 0.5 only to classify errors.
# Precision@10 remains the ranking metric.
error_analysis["predicted"] = (
    error_analysis["model_score"] >= 0.5
).astype(int)


# ---------------------------------------------------------
# 2. Identify false positives and false negatives
# ---------------------------------------------------------

false_positives = error_analysis[
    (error_analysis["predicted"] == 1) &
    (error_analysis["actual"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

false_negatives = error_analysis[
    (error_analysis["predicted"] == 0) &
    (error_analysis["actual"] == 1)
].sort_values(
    "model_score",
    ascending=True
)

print("=== Error Analysis ===")
print(f"False positives: {len(false_positives):,}")
print(f"False negatives: {len(false_negatives):,}")


# ---------------------------------------------------------
# 3. Show what the model relies on
# ---------------------------------------------------------

print("\n=== Top 3 Model Features ===")
display(top3_features)


# ---------------------------------------------------------
# 4. Compare characteristics of the two error types
# ---------------------------------------------------------

error_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]

error_summary = pd.DataFrame({
    "False positives": false_positives[error_features].mean(),
    "False negatives": false_negatives[error_features].mean()
})

print("\n=== Error Profile ===")
display(error_summary)


# ---------------------------------------------------------
# 5. Inspect highest-confidence mistakes
# ---------------------------------------------------------

display_columns = [
    "content_id",
    "client_id",
    "model_score",
    "actual",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]

print("\n=== Top 5 False Positives ===")
display(false_positives[display_columns].head(5))

print("\n=== Top 5 False Negatives ===")
display(false_negatives[display_columns].head(5))


# ---------------------------------------------------------
# 6. Inspect the model's top-ranked items
# ---------------------------------------------------------

model_ranked = test_df.copy()

model_ranked["model_score"] = model_scores
model_ranked["actual_declining"] = y_test.to_numpy()

model_ranked = (
    model_ranked
    .sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)

model_ranked["model_rank"] = np.arange(len(model_ranked)) + 1

print("\n=== Top 10 Model-Ranked Items ===")

display(
    model_ranked.head(10)[
        [
            "model_rank",
            "content_id",
            "client_id",
            "model_score",
            "actual_declining",
            "impressions_90d",
            "avg_position",
            "ctr",
            "content_age_days"
        ]
    ]
)


# ---------------------------------------------------------
# 7. Final feature audit
# ---------------------------------------------------------

FORBIDDEN_FEATURES = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
}

forbidden_used = [
    col for col in feature_cols
    if col in FORBIDDEN_FEATURES
]

print("\n=== Final Feature Audit ===")
print("Model features:", len(feature_cols))
print("Forbidden features used:", forbidden_used)

assert len(forbidden_used) == 0

print("✓ No forbidden features used.")

=== Error Analysis ===
False positives: 1,606
False negatives: 1,415

=== Top 3 Model Features ===


,feature,importance
4,numeric__char_count,0.167367
3,numeric__word_count,0.163929
5,numeric__content_age_days,0.138457



=== Error Profile ===


,False positives,False negatives
impressions_90d,4154.468867,3340.250177
avg_position,17.173910,14.618233
ctr,0.282223,0.291039
content_age_days,281.671856,314.874205
days_since_last_update,44.309465,24.727208



=== Top 5 False Positives ===


...


=== Top 5 False Negatives ===


,content_id,client_id,model_score,actual,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update
4476,content_9e390d63ef52,client_4e07408562,0.02500,1,1035,15.6,0.19,390,7
9770,content_da284c9673e7,client_e629fa6598,0.02829,1,321,24.4,0.62,460,22
15530,content_20a46ffa0f35,client_e629fa6598,0.02829,1,404,24.2,0.00,460,22
8113,content_60aec4203852,client_e629fa6598,0.02829,1,956,45.0,0.00,460,22
5017,content_d8a3c9484a1b,client_e629fa6598,0.02829,1,523,36.5,0.00,460,22



=== Top 10 Model-Ranked Items ===


,model_rank,content_id,client_id,model_score,actual_declining,impressions_90d,avg_position,ctr,content_age_days
0,1,content_268910e38131,client_8527a891e2,0.995,1,371,9.3,0.27,275
1,2,content_b658db887566,client_8527a891e2,0.995,0,13,8.4,0.00,237
2,3,content_c6cffac60462,client_8527a891e2,0.995,1,39,4.6,0.00,275
3,4,content_85a09b3e3147,client_8527a891e2,0.990,1,116,6.4,0.00,237
4,5,content_97fd095ebfd1,client_8527a891e2,0.990,0,39,8.4,0.00,275
5,6,content_287466584c52,client_4e07408562,0.990,1,1668,20.9,0.12,277
6,7,content_f49660e074e9,client_8527a891e2,0.990,0,1432,20.2,0.35,223
7,8,content_765d5a7d6ef2,client_8527a891e2,0.990,0,10,7.3,0.00,275
8,9,content_1d4d78ba6371,client_8527a891e2,0.990,1,11,7.1,0.00,238
9,10,content_33e1340be022,client_8527a891e2,0.985,0,1,5.0,0.00,223



=== Final Feature Audit ===
Model features: 19
Forbidden features used: []
✓ No forbidden features used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.